# XGBoost Spark — predicción de puntajes ICFES

Regresión distribuida con [`SparkXGBRegressor`](https://xgboost.readthedocs.io/en/latest/tutorials/spark_estimator.html) sobre datos en HDFS.

- **Targets:** los seis puntajes Saber 11 (`punt_global`, `punt_matematicas`, …).
- **Features:** variables de familia, estudiante, colegio + indicadores municipales (`accesos_por_habitante`, `cobertura_neta`, …).
- **Métricas (test):** RMSE, MAE, R².
- **Script CLI:** `python ml_train.py --sample 100000` (desarrollo) o sin `--sample` (producción en clúster).

In [ ]:
import os

# Desarrollo: muestra; producción: SAMPLE_N=None y SPARK_MASTER_URL=spark://spark-master:7077
SAMPLE_N = int(os.environ.get("ML_SAMPLE_N", "50000"))
TRAIN_ALL = os.environ.get("ML_TRAIN_ALL", "0") == "1"

from ml_train import (
    ICFES_SCORE_COLS,
    build_ml_dataset,
    build_ml_spark_session,
    predict_scores,
    print_metrics_table,
    train_all_targets,
    train_one_target,
)

spark = build_ml_spark_session()
spark.sparkContext.setLogLevel("WARN")

In [ ]:
df_ml = build_ml_dataset(spark, sample_n=SAMPLE_N)
n_rows = df_ml.count()
print(f"Filas ML: {n_rows:,}")
df_ml.select("has_muni_features").groupBy("has_muni_features").count().show()

## Entrenamiento (piloto o los 6 targets)

Split **70% train / 15% val / 15% test**. Early stopping en validación. Modelos en `hdfs://spark-master:9000/data/models/xgb_{target}/`.

In [ ]:
if TRAIN_ALL:
    results = train_all_targets(spark, df_ml)
else:
    results = [train_one_target(spark, df_ml, "punt_global")]

print_metrics_table(results)

## Inferencia: dato nuevo → puntaje predicho

Misma estructura de columnas que el Parquet ICFES (familia, colegio, municipio, periodo). Las predicciones se acotan a [0, 500].

In [ ]:
nuevo_estudiante = {
    "fami_tieneinternet": "Si",
    "fami_estratovivienda": "2",
    "fami_educacionmadre": "Superior universitario",
    "fami_educacionpadre": "Tecnico profesional completo",
    "estu_genero": "F",
    "cole_naturaleza": "OFICIAL",
    "cole_jornada": "COMPLETA",
    "cole_area_ubicacion": "URBANO",
    "cole_bilingue": "NO",
    "cole_calendario": "CALENDARIO A",
    "cole_cod_mcpio_ubicacion": "11001",
    "periodo": "20221",
    "accesos_por_habitante": 0.35,
    "total_accesos_internet": 120000.0,
    "cobertura_neta": 85.0,
    "departamento": "BOGOTA",
}

predicciones = predict_scores(spark, nuevo_estudiante)
for k, v in sorted(predicciones.items()):
    print(f"{k}: {v:.1f}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Gráfico predicción vs real (muestra del último modelo entrenado en la celda anterior)
pred_pdf = (
    results[-1]["test_predictions"]
    .select("label", "prediction")
    .sample(False, 0.05, seed=42)
    .toPandas()
)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(pred_pdf["label"], pred_pdf["prediction"], alpha=0.3, s=8)
ax.plot([0, 500], [0, 500], "r--", label="ideal")
ax.set_xlabel("Puntaje real")
ax.set_ylabel("Predicción")
ax.set_title(f"Test: {results[-1]['target']}")
ax.legend()
plt.tight_layout()
plt.show()

## Perfilamiento Socioeconómico (K-Means + Random Forest)

En esta sección utilizamos el método del codo con K-Means para encontrar agrupaciones basadas en variables socioeconómicas (incluyendo estratos, educación de padres, internet y cuartos en el hogar) junto con los puntajes.

Además, usamos Random Forest para obtener la importancia de variables al predecir el puntaje global, identificando así el efecto del internet frente a otros mediadores territoriales.

In [ ]:
from ml_profiling import build_profiling_dataset, PROFILING_CAT_FEATURES, PROFILING_NUM_FEATURES, find_optimal_k_elbow, train_random_forest_importance

# Construimos un dataset especial para perfilamiento que incluye las variables solicitadas.
df_prof = build_profiling_dataset(spark, sample_n=SAMPLE_N)
print(f"Filas Profiling: {df_prof.count():,}")

In [ ]:
# 1. Método del Codo para K-Means (Evaluando Silhouette Score)
k_vals, scores = find_optimal_k_elbow(df_prof, PROFILING_CAT_FEATURES, PROFILING_NUM_FEATURES, max_k=8)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(k_vals, scores, marker="o", linestyle="--", color="b")
ax.set_title("Método de la Silueta para K-Means")
ax.set_xlabel("Número de Clusters (k)")
ax.set_ylabel("Silhouette Score")
ax.grid(True)
plt.show()

In [ ]:
# 2. Importancia de Variables (Feature Importance) con Random Forest
importances_df = train_random_forest_importance(df_prof, PROFILING_CAT_FEATURES, PROFILING_NUM_FEATURES, target_col="punt_global")

fig, ax = plt.subplots(figsize=(8, 5))
importances_df.set_index("Feature").plot(kind="bar", ax=ax, color="skyblue")
ax.set_title("Importancia de Variables (Random Forest)")
ax.set_ylabel("Importance")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
df_ml.unpersist()
spark.stop()